# 02 - Preprocessing
Create a clean regular time series and save it under `data/processed/`.

In [23]:
import sys
from pathlib import Path
import pandas as pd
sys.path.append('..')

df = pd.read_csv('../data/raw/retail_store_inventory.csv')


#from src.preprocessing import load_data, prepare_time_series, save_processed_data

#df = load_data('../data/raw/retail_store_inventory.csv')
#ts = prepare_time_series(df, date_col='Date', target_col='Sales')
#save_processed_data(ts, '../data/processed/daily_sales.csv')
#ts.head()

In [67]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)
pd.set_option('display.width', 500)

In [ ]:
# train test split yapılmadan önce yapulacaklar:
    # Tarihten Year, Month, Day, DayOfWeek veya Seasonality_NEW türetme.
    # Hatalı gürültü değişkenini (Seasonality) düşürme.
    # Satır bazlı matematiksel işlemler: Net_Price = Price * (1 - Discount/100) veya Price_Diff.
    # Demand Forecast sütunundaki negatif değerleri (-9.99) minimum 0 yapma.

In [24]:
# Date kolonunu objectten datetime64[ns] tipine çevirme
df['Date'] = df['Date'].astype('datetime64[ns]')

In [25]:
# Zaman Serisi Kronolojik Sıralama
# Sıralama bozuk olursa, shift(1) işlemi yanlış güne denk gelebilir
df = df.sort_values(by=['Store ID', 'Product ID', 'Date']).reset_index(drop=True)

In [26]:
# Seasonality kolonu artık gereksiz ve gürültü değişkeni olduğu için düşürülüyor.
df.drop(["Seasonality"], axis=1, inplace=True)

In [27]:
# Demand Forecast'i dropla data leakage'a sebep oluyor
df = df.drop(columns=['Demand Forecast'])

In [ ]:
# Year - Month - Day- DayOfWeek feature'larını türetme
import datetime as dt
def date_features(dataframe):
    dataframe['Year'] = dataframe['Date'].dt.year
    dataframe['Month'] = dataframe['Date'].dt.month
    dataframe['Day'] = dataframe['Date'].dt.day
    dataframe['DayOfWeek'] = dataframe['Date'].dt.dayofweek
    return dataframe

df = date_features(df)

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Year,Month,Day,DayOfWeek
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,33.50,20,Rainy,0,29.69,2022,1,1,5
1,2022-01-02,S001,P0001,Groceries,West,116,81,104,27.95,10,Cloudy,0,30.89,2022,1,2,6
2,2022-01-03,S001,P0001,Electronics,West,154,5,189,62.70,20,Rainy,0,58.22,2022,1,3,0
3,2022-01-04,S001,P0001,Groceries,South,85,58,193,77.88,15,Cloudy,1,75.99,2022,1,4,1
4,2022-01-05,S001,P0001,Groceries,South,238,147,37,28.46,20,Sunny,1,29.40,2022,1,5,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73095,2023-12-28,S005,P0020,Groceries,South,198,56,27,21.75,5,Sunny,1,25.29,2023,12,28,3
73096,2023-12-29,S005,P0020,Clothing,East,446,268,30,85.58,20,Sunny,1,87.63,2023,12,29,4
73097,2023-12-30,S005,P0020,Toys,North,251,149,181,79.48,10,Cloudy,1,82.69,2023,12,30,5
73098,2023-12-31,S005,P0020,Furniture,East,64,40,99,90.79,5,Snowy,1,91.67,2023,12,31,6


In [ ]:
# Date'den verileri türettikten sonra Date'i dropla
df.drop(["Date"], axis=1, inplace=True)

In [29]:
#Seanolity sentetik bir şekilde rastgele üretilmişti. 
# Bu nedenle, gerçek aya göre türetmek daha mantıklı olabilir.

df["Seasonality_NEW"] = df["Date"].dt.month.map({
                                                12: "Winter", 1: "Winter", 2: "Winter",
                                                3: "Spring", 4: "Spring", 5: "Spring",
                                                6: "Summer", 7: "Summer", 8: "Summer",
                                                9: "Fall", 10: "Fall", 11: "Fall"})

In [30]:
# Price'a Discount uygulanarak Net_Price hesaplanması
df["NET_PRICE"] = df["Price"] * (1 - df["Discount"]/100)

In [31]:
# Lag feature'ları türet
# Store ve Product ID kırılımına göre üret böylece o mağazada o ürünün geçmiş s
# atışları ile tahmin yapılabilir.

def lag_features(dataframe, lags):
    for lag in lags:
        dataframe['units_sold_lag_' + str(lag)] = dataframe.groupby(["Store ID", "Product ID"])['Units Sold'].transform(
            lambda x: x.shift(lag)) 
    return dataframe

df = lag_features(df, [1, 2, 3, 7, 14, 30, 365])

In [32]:
# rolling mean feature'ları türet
def roll_mean_features(dataframe, windows):
    for window in windows:
        dataframe['sales_roll_mean_' + str(window)] = (
            dataframe.groupby(["Store ID", "Product ID"])['Units Sold']
            .transform(lambda x: x.shift(1).rolling(window=window).mean())
        )
    return dataframe
df = roll_mean_features(df, [7, 30])

In [33]:
df.head()

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,...,NET_PRICE,units_sold_lag_1,units_sold_lag_2,units_sold_lag_3,units_sold_lag_7,units_sold_lag_14,units_sold_lag_30,units_sold_lag_365,sales_roll_mean_7,sales_roll_mean_30
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,33.50,20,...,26.800,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2022-01-02,S001,P0001,Groceries,West,116,81,104,27.95,10,...,25.155,127.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2022-01-03,S001,P0001,Electronics,West,154,5,189,62.70,20,...,50.160,81.0,127.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2022-01-04,S001,P0001,Groceries,South,85,58,193,77.88,15,...,66.198,5.0,81.0,127.0,NaN,NaN,NaN,NaN,NaN,NaN
4,2022-01-05,S001,P0001,Groceries,South,238,147,37,28.46,20,...,22.768,58.0,5.0,81.0,NaN,NaN,NaN,NaN,NaN,NaN


Observations: 73100
Variables: 28
cat_cols: 10
num_cols: 18
cat_but_car: 0
num_but_cat: 4


In [40]:
df["Discount"].value_counts()

20    14715
0     14662
15    14624
5     14591
10    14508
Name: Discount, dtype: int64

In [53]:
cat_cols = ['Store ID', 'Product ID', 'Category', 'Region', 'Weather Condition', 'Seasonality_NEW', 'Holiday/Promotion']

num_cols = [ 'Discount','Inventory Level', 'Units Ordered', 'Price', 'Competitor Pricing', 'Month', 'Day', 'NET_PRICE', 'units_sold_lag_1', 'units_sold_lag_2', 'units_sold_lag_3', 'units_sold_lag_7', 'units_sold_lag_14', 'units_sold_lag_30', 'units_sold_lag_365', 'sales_roll_mean_7', 'sales_roll_mean_30', 'Year', 'DayOfWeek']

In [54]:
print(f"Categorical Columns: {cat_cols}")
print(f"Numerical Columns: {num_cols}")

Categorical Columns: ['Store ID', 'Product ID', 'Category', 'Region', 'Weather Condition', 'Seasonality_NEW', 'Holiday/Promotion']
Numerical Columns: ['Discount', 'Inventory Level', 'Units Ordered', 'Price', 'Competitor Pricing', 'Month', 'Day', 'NET_PRICE', 'units_sold_lag_1', 'units_sold_lag_2', 'units_sold_lag_3', 'units_sold_lag_7', 'units_sold_lag_14', 'units_sold_lag_30', 'units_sold_lag_365', 'sales_roll_mean_7', 'sales_roll_mean_30', 'Year', 'DayOfWeek']


In [57]:
# Test ve train setlerini ayır 
    # Bu bir zaman serisi tahmin problemidir. Bu nedenle, verileri 
    # rastgele ayırmak yerine, belirli bir tarih noktasına göre ayırmak gereklidir

# Dataset 2022-01-01 - 2024-01-01 arasını kapsar. Son 30 gün test seti olarak ayırmak mantıklı olabilir. 
# Bu nedenle, 2023-12-01 tarihinden itibaren olan veriler test seti olarak ayrılabilir.

train = df[df['Date'] < '2023-12-01']
test = df[df['Date'] >= '2023-12-01']

In [56]:
train.shape, test.shape

((69900, 28), (3200, 28))

In [58]:
df["Category"].value_counts()

Furniture      14699
Toys           14643
Clothing       14626
Groceries      14611
Electronics    14521
Name: Category, dtype: int64

In [59]:
# One-Hot-Encoding ID'lere yapılmamalı, kardinalitisi çok yüksek
# Curse of Dimensionality sorunu yaratır, Bu yüzden OHE yapılacak kategorik değişkenleri
# seçtik

cat_cols_OHE = ['Category', 'Region', 'Weather Condition', 'Seasonality_NEW']

In [ ]:
# Train test olarak ayırıldıktan sonra ilk olarak cat collara one hot yapılacak


#  Train ve Test kümesinde get_dummies uygula
train_ohe_df = pd.get_dummies(train[cat_cols_OHE], drop_first=True)
test_ohe_df = pd.get_dummies(test[cat_cols_OHE], drop_first=True)

#  Trainde olan her şey gelicek, test'te var olanlar gelicek olmayanlar drop olcak
# Yani referans olarak train seti alınacak. Böylecce train testi görmemiş olacak
train_ohe_df, test_ohe_df = train_ohe_df.align(test_ohe_df, join='left', axis=1, fill_value=0)

# 3. Eski kategorik metin kolonlarını drop edip yenileriyle birleştir
final_train_df = pd.concat([train.drop(columns=cat_cols_OHE), train_ohe_df], axis=1)
final_test_df = pd.concat([test.drop(columns=cat_cols_OHE), test_ohe_df], axis=1)

Dummy variable 1 sütun düşürür. OHE yapılırken 1 eksik column oluşur yani hepsi 0 0 0
ise yazılmayan sütundur. 
Region_East, Region_West, Region_North hepsi 0 ise demek ki Southtır. kolon sayısı artmasın diye South için ayrı kolon oluşmaz.

!!!!! Bunun için 40 saat düşündüm !!!!

In [92]:
cat_features = ['Store ID', 'Product ID']

# ID'ler cardinality olarak çoktur OHE yapılamaz ama eğitim ve test için de gereklidir.
# Label Encoding 0-1-2 gibi ordinal değer atar ama ID'ler birbirinden üstün değildir
# LightGBM gibi modeller için dönüşüm şarttır bu yüzden bu veriler categoric olarak atanır
# arka planda Pandas her ID'ye  bir tamsayı kodu atar
for col in cat_features:
    final_train_df[col] = final_train_df[col].astype('category')
    # test setini train'in kategori kümesine göre hizala
    final_test_df[col] = pd.Categorical(
        final_test_df[col],
        categories=final_train_df[col].cat.categories
    )


In [93]:
for col in cat_features:
    unseen = set(final_test_df[col].dropna().unique()) - set(final_train_df[col].cat.categories)
    if unseen:
        print(f"{col} için train'de görülmemiş değerler: {unseen}")

In [87]:
# Outliers'ı Baskılama
def outlier_thresholds(dataframe, col_name, q1 = 0.25, q3 = 0.75):

    quartile1 = dataframe[col_name].quantile(q1)
    quartile3 = dataframe[col_name].quantile(q3)

    iqr = quartile3 - quartile1
    upper_limit = quartile3 + 1.5 * iqr
    lower_limit = quartile1 - 1.5 * iqr

    return lower_limit, upper_limit

def check_outliers( dataframe, col_name):
    lower_limit, upper_limit = outlier_thresholds(dataframe, col_name)
    if dataframe[(dataframe[col_name] > upper_limit) | (dataframe[col_name] < lower_limit)].any(axis=None):
        return True
    else:
        return False

def suppress_outliers(dataframe, col_name):
    low_limit, upper_limit = outlier_thresholds(dataframe, col_name)
    dataframe.loc[(dataframe[col_name] < low_limit), col_name] = low_limit
    dataframe.loc[(dataframe[col_name] > upper_limit), col_name] = upper_limit

In [90]:
num_cols

['Discount',
 'Inventory Level',
 'Units Ordered',
 'Price',
 'Competitor Pricing',
 'Month',
 'Day',
 'NET_PRICE',
 'units_sold_lag_1',
 'units_sold_lag_2',
 'units_sold_lag_3',
 'units_sold_lag_7',
 'units_sold_lag_14',
 'units_sold_lag_30',
 'units_sold_lag_365',
 'sales_roll_mean_7',
 'sales_roll_mean_30',
 'Year',
 'DayOfWeek']

In [89]:
for col in num_cols:
    if check_outliers(final_train_df, col):
        print(f"{col} has outliers")

units_sold_lag_1 has outliers
units_sold_lag_2 has outliers
units_sold_lag_3 has outliers
units_sold_lag_7 has outliers
units_sold_lag_14 has outliers
units_sold_lag_30 has outliers
units_sold_lag_365 has outliers
sales_roll_mean_7 has outliers
sales_roll_mean_30 has outliers


In [91]:
final_train_df.head()

,Date,Store ID,Product ID,Inventory Level,Units Sold,Units Ordered,Price,Discount,Holiday/Promotion,Competitor Pricing,Year,Month,Day,DayOfWeek,NET_PRICE,units_sold_lag_1,units_sold_lag_2,units_sold_lag_3,units_sold_lag_7,units_sold_lag_14,units_sold_lag_30,units_sold_lag_365,sales_roll_mean_7,sales_roll_mean_30,Category_Electronics,Category_Furniture,Category_Groceries,Category_Toys,Region_North,Region_South,Region_West,Weather Condition_Rainy,Weather Condition_Snowy,Weather Condition_Sunny,Seasonality_NEW_Spring,Seasonality_NEW_Summer,Seasonality_NEW_Winter
0,2022-01-01,S001,P0001,231,127,55,33.500,20,0,29.690,2022,1,1,5,26.800,nan,nan,nan,nan,nan,nan,nan,nan,nan,0,0,1,0,1,0,0,1,0,0,0,0,1
1,2022-01-02,S001,P0001,116,81,104,27.950,10,0,30.890,2022,1,2,6,25.155,127.000,nan,nan,nan,nan,nan,nan,nan,nan,0,0,1,0,0,0,1,0,0,0,0,0,1
2,2022-01-03,S001,P0001,154,5,189,62.700,20,0,58.220,2022,1,3,0,50.160,81.000,127.000,nan,nan,nan,nan,nan,nan,nan,1,0,0,0,0,0,1,1,0,0,0,0,1
3,2022-01-04,S001,P0001,85,58,193,77.880,15,1,75.990,2022,1,4,1,66.198,5.000,81.000,127.000,nan,nan,nan,nan,nan,nan,0,0,1,0,0,1,0,0,0,0,0,0,1
4,2022-01-05,S001,P0001,238,147,37,28.460,20,1,29.400,2022,1,5,2,22.768,58.000,5.000,81.000,nan,nan,nan,nan,nan,nan,0,0,1,0,0,1,0,0,0,1,0,0,1
